# Post-Quantum Readiness for a Synthetic Clinic

A student project for the IBM Quantum Qiskit Hackathon at Miami-Dade College (Qiskit Fall Fest 2026).

**Question this notebook asks:** If a small clinic wanted to prepare for future quantum threats to public-key cryptography, which uses of cryptography should it review first, and why?

The clinic in this project is fictional. I call it Bayside Community Clinic. Every system, algorithm choice, and "how long must this stay secret" value is a made-up example. This notebook does **not** describe Miami-Dade College or any real clinic, patient, or network.

I am a second-year cybersecurity student, new to quantum computing. The quantum part of this notebook is a small teaching demo of Shor's period-finding routine, which is the method that would threaten RSA and elliptic-curve cryptography if a much more capable quantum computer existed. I ran it on a simulator and on real IBM hardware. It factors the number 15. That is a classroom example, not an attack on real encryption, and it is not a claim of quantum advantage.

The migration ranking uses a rule I can explain in plain language. It is **not** an official NIST, CISA, or industry score.


## Synthetic Crypto inventory and an explainable ranking

A **crypto inventory** is just a list of where cryptography is used: on a website, in email, in backups, and so on. Without that list, a clinic cannot tell what to migrate first.

The table is built in the next cell from 12 synthetic assets. The ranking rule is visible on purpose:

| Factor | 0 | 1 | 2 |
| --- | --- | --- | --- |
| Quantum-vulnerable? (RSA, elliptic-curve, Diffie-Hellman) | No | Partly (for example AES plus an RSA key wrap) | Yes |
| How long the data must stay secret | Under 2 years | 2 to 7 years | Over 7 years |
| Who can reach the system | Internal | Partner | Internet |
| What the crypto is doing | Other | Signature | Key exchange or key transport |

**Priority score** = the four numbers added together (0 to 8). If two assets tie, the one that looks easier to migrate is listed first. **Now** means 7 or 8, **Next** means 4 to 6, and **Later** means 0 to 3.

Key exchange and key transport get the higher role score because of **harvest now, decrypt later**: someone could copy encrypted data today and try to decrypt it in the future. Signatures matter too, but they are a different kind of risk.

**AES is not in the same bucket as RSA.** A different quantum method, Grover's algorithm, can speed up brute-force guessing against symmetric ciphers. The usual response is a longer key, such as AES-256. I do not treat AES-256 as needing a swap to ML-KEM or ML-DSA the way RSA and elliptic-curve tools do, and I do not claim AES-256 is "quantum-proof."

The years and "easy / medium / hard" labels are invented assumptions for this demo, not legal retention rules and not a real vendor assessment.

Suggested replacements follow NIST's August 13, 2024 standards: **FIPS 203 (ML-KEM)** for key establishment, **FIPS 204 (ML-DSA)** for signatures, and **FIPS 205 (SLH-DSA)** as a conservative signature backup. NIST allows a hybrid setup that combines a classical method with a post-quantum one. Federal high-impact deadlines mentioned later in this notebook are background only. They are not requirements I am placing on this fictional clinic.

In [1]:
# SYNTHETIC inventory and a transparent priority rule.
# This cell does not describe any real clinic, patient, or MDC system.
# The scores below are our team's ranking rule, not an official NIST or CISA score.

import pandas as pd

print(
    "SYNTHETIC DATA DISCLAIMER: Every row is a made-up example for a student "
    "hackathon project about a fictional clinic named Bayside Community Clinic. "
    "It does not describe Miami-Dade College or any real organization, patient, or system."
)
print()
print("Our ranking rule (not an official score)")
print("  Quantum-vulnerable: No=0, Partly=1 (for example AES plus RSA key wrap), Yes=2")
print("  How long the data must stay secret: under 2 years=0, 2 to 7 years=1, over 7 years=2")
print("  Who can reach it: Internal=0, Partner=1, Internet=2")
print("  What the crypto is doing: Other=0, Signature=1, Key exchange or key transport=2")
print("  Priority score = the four numbers added together (0 to 8).")
print("  If two assets tie, the one that looks easier to migrate is listed first.")
print("  Tiers: Now = 7 or 8, Next = 4 to 6, Later = 0 to 3.")
print()

# confidentiality_years and other values are invented assumptions, not legal retention rules.
rows = [
    {
        "asset": "Patient portal (TLS)",
        "purpose": "Establish a secure web connection",
        "algorithm": "ECDHE + RSA certificate",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 10,
        "exposure": "Internet",
        "role": "Key exchange",
        "migration_ease": "Medium",
        "migration_notes": "Ask the vendor about hybrid TLS with ML-KEM (FIPS 203).",
    },
    {
        "asset": "Staff VPN",
        "purpose": "Protect remote staff access",
        "algorithm": "RSA-2048 (IKE)",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 5,
        "exposure": "Internet",
        "role": "Key exchange",
        "migration_ease": "Medium",
        "migration_notes": "Check the VPN vendor's post-quantum or hybrid roadmap.",
    },
    {
        "asset": "Lab partner gateway (TLS)",
        "purpose": "Share lab results with partner labs",
        "algorithm": "ECDHE + RSA certificate",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 10,
        "exposure": "Partner",
        "role": "Key exchange",
        "migration_ease": "Medium",
        "migration_notes": "Coordinate with the lab vendor; hybrid ML-KEM if supported.",
    },
    {
        "asset": "Email (S/MIME)",
        "purpose": "Encrypt and sign clinical email",
        "algorithm": "RSA-2048",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 7,
        "exposure": "Internet",
        "role": "Key transport",
        "migration_ease": "Hard",
        "migration_notes": "The encryption part is the harvest-now concern; plan signatures too.",
    },
    {
        "asset": "Insurance claims connection",
        "purpose": "Send claims to a payer",
        "algorithm": "RSA-2048 (TLS)",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 7,
        "exposure": "Partner",
        "role": "Key exchange",
        "migration_ease": "Medium",
        "migration_notes": "Depends on the payer's TLS support for ML-KEM.",
    },
    {
        "asset": "Patient document signing",
        "purpose": "Sign discharge and consent forms",
        "algorithm": "RSA-2048",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 7,
        "exposure": "Internet",
        "role": "Signature",
        "migration_ease": "Hard",
        "migration_notes": "Move new signatures to ML-DSA (FIPS 204); keep a way to check old ones.",
    },
    {
        "asset": "Telehealth sessions",
        "purpose": "Encrypt live video visits",
        "algorithm": "ECDHE",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 1,
        "exposure": "Internet",
        "role": "Key exchange",
        "migration_ease": "Medium",
        "migration_notes": "Sessions are short-lived, but the service is still internet-facing.",
    },
    {
        "asset": "Backup storage",
        "purpose": "Encrypt stored backups and wrap the keys",
        "algorithm": "AES-256 with RSA key wrap",
        "quantum_vulnerable": "Partly",
        "confidentiality_years": 10,
        "exposure": "Internal",
        "role": "Key transport",
        "migration_ease": "Medium",
        "migration_notes": "AES-256 can stay; migrate the RSA wrap to ML-KEM (FIPS 203).",
    },
    {
        "asset": "Intranet app code signing",
        "purpose": "Verify an internal app has not been altered",
        "algorithm": "ECDSA P-256",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 2,
        "exposure": "Internal",
        "role": "Signature",
        "migration_ease": "Hard",
        "migration_notes": "Internal only; plan how older signed builds will still verify.",
    },
    {
        "asset": "Admin SSH to Wi-Fi controllers",
        "purpose": "Authenticate administrators",
        "algorithm": "Ed25519",
        "quantum_vulnerable": "Yes",
        "confidentiality_years": 1,
        "exposure": "Internal",
        "role": "Signature",
        "migration_ease": "Easy",
        "migration_notes": "Short-lived internal access, but Ed25519 is still quantum-vulnerable.",
    },
    {
        "asset": "EHR database at rest",
        "purpose": "Encrypt stored medical records",
        "algorithm": "AES-256",
        "quantum_vulnerable": "No",
        "confidentiality_years": 10,
        "exposure": "Internal",
        "role": "Other",
        "migration_ease": "Easy",
        "migration_notes": "Not the same threat as RSA or elliptic-curve crypto; keep AES-256 and review key management.",
    },
    {
        "asset": "Staff laptop disk encryption",
        "purpose": "Encrypt data if a laptop is lost or stolen",
        "algorithm": "AES-256",
        "quantum_vulnerable": "No",
        "confidentiality_years": 5,
        "exposure": "Internal",
        "role": "Other",
        "migration_ease": "Easy",
        "migration_notes": "Control row: AES-256 is not treated as a post-quantum algorithm swap.",
    },
]

QV = {"No": 0, "Partly": 1, "Yes": 2}
EXPOSURE = {"Internal": 0, "Partner": 1, "Internet": 2}
EASE_RANK = {"Easy": 0, "Medium": 1, "Hard": 2}  # lower means easier; used only to break ties


def secrecy_score(years):
    if years < 2:
        return 0
    if years <= 7:
        return 1
    return 2


def role_score(role):
    # Key transport is grouped with key exchange because both can leak data later
    # if today's ciphertext is stored and decrypted in the future.
    if role in ("Key exchange", "Key transport"):
        return 2
    if role == "Signature":
        return 1
    return 0


def suggested_path(row):
    if row["quantum_vulnerable"] == "No":
        return "Keep AES-256; review how keys are stored. No FIPS 203/204/205 swap for the bulk cipher."
    if "S/MIME" in row["asset"]:
        return "FIPS 203 (ML-KEM) for the encryption part; FIPS 204 (ML-DSA) for signatures. Hybrid allowed."
    if row["role"] in ("Key exchange", "Key transport"):
        return "FIPS 203 (ML-KEM); a hybrid classical-plus-PQC setup is allowed."
    if row["role"] == "Signature":
        return "FIPS 204 (ML-DSA); FIPS 205 (SLH-DSA) is a conservative backup."
    return "Review case by case."


def tier_for(score):
    if score >= 7:
        return "Now"
    if score >= 4:
        return "Next"
    return "Later"


def reason(row):
    qv = row["quantum_vulnerable"]
    years = row["confidentiality_years"]
    if years < 2:
        secret = f"data is assumed to need secrecy for under 2 years ({years})"
    elif years <= 7:
        secret = f"data is assumed to need secrecy for 2 to 7 years ({years})"
    else:
        secret = f"data is assumed to need secrecy for more than 7 years ({years})"
    bits = [
        f"quantum-vulnerable crypto: {qv} ({row['algorithm']})",
        secret,
        f"exposure: {row['exposure']}",
        f"role: {row['role']}",
    ]
    return (
        f"Score {row['priority_score']} ({row['tier']}). "
        f"Reasons: {'; '.join(bits)}. "
        f"Suggested path: {row['suggested_path']}"
    )


df = pd.DataFrame(rows)
df["s_qv"] = df["quantum_vulnerable"].map(QV)
df["s_secret"] = df["confidentiality_years"].map(secrecy_score)
df["s_exposure"] = df["exposure"].map(EXPOSURE)
df["s_role"] = df["role"].map(role_score)
df["priority_score"] = df["s_qv"] + df["s_secret"] + df["s_exposure"] + df["s_role"]
df["tier"] = df["priority_score"].map(tier_for)
df["ease_rank"] = df["migration_ease"].map(EASE_RANK)
df["suggested_path"] = df.apply(suggested_path, axis=1)
df["reason"] = df.apply(reason, axis=1)

ranked = df.sort_values(
    by=["priority_score", "ease_rank", "asset"],
    ascending=[False, True, True],
).reset_index(drop=True)
ranked.insert(0, "rank", range(1, len(ranked) + 1))

show = ranked[
    [
        "rank",
        "asset",
        "algorithm",
        "priority_score",
        "tier",
        "migration_ease",
        "suggested_path",
    ]
]
print(show.to_string(index=False))
print()
print("Why each asset received its score")
for _, row in ranked.iterrows():
    print(f"{row['rank']}. {row['asset']}: {row['reason']}")

# A simple check: if we ignored exposure, would the top 3 change?
alt = df.copy()
alt["alt_score"] = alt["s_qv"] + alt["s_secret"] + alt["s_role"]
alt_ranked = alt.sort_values(
    by=["alt_score", "ease_rank", "asset"],
    ascending=[False, True, True],
)
top_default = list(ranked.head(3)["asset"])
top_alt = list(alt_ranked.head(3)["asset"])
print()
print("Sensitivity check (not a second official ranking)")
print("Top 3 with our full rule:", ", ".join(top_default))
print("Top 3 if we ignore exposure:", ", ".join(top_alt))
if top_default == top_alt:
    print("The top 3 stayed the same in this check.")
else:
    print(
        "The very top items are close, but mid-ranked order can change if we drop a factor. "
        "That is expected, and it is a reason to show the rule instead of hiding it."
    )

SYNTHETIC DATA DISCLAIMER: Every row is a made-up example for a student hackathon project about a fictional clinic named Bayside Community Clinic. It does not describe Miami-Dade College or any real organization, patient, or system.

Our ranking rule (not an official score)
  Quantum-vulnerable: No=0, Partly=1 (for example AES plus RSA key wrap), Yes=2
  How long the data must stay secret: under 2 years=0, 2 to 7 years=1, over 7 years=2
  Who can reach it: Internal=0, Partner=1, Internet=2
  What the crypto is doing: Other=0, Signature=1, Key exchange or key transport=2
  Priority score = the four numbers added together (0 to 8).
  If two assets tie, the one that looks easier to migrate is listed first.
  Tiers: Now = 7 or 8, Next = 4 to 6, Later = 0 to 3.

 rank                          asset                 algorithm  priority_score  tier migration_ease                                                                               suggested_path
    1           Patient portal (TLS)   

## What Shor's algorithm is doing

Public-key tools such as RSA are hard to break classically because factoring a very large number, or solving a related math problem, is extremely difficult. Shor's algorithm is a quantum method that can find a hidden repeating pattern, called a **period**, and a ordinary computer can then turn that period into factors.

In this notebook I use a tiny example: N = 15 and a helper value a = 7. The circuit tries to learn the period of multiplying by 7, modulo 15. From some of those results, classical post-processing recovers the factors **3 × 5**.

This is a mechanism demo. It shows *why* people worry about RSA and elliptic-curve cryptography. It does **not** break a real key. Real RSA numbers are hundreds of digits long, and I am not claiming that today's quantum computers can do that, or that they will be able to do it by a certain date.

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime

In [ ]:
import qiskit
print("Qiskit version:", qiskit.__version__)


Qiskit version: 2.6.2


## Building a small period-finding circuit

The circuit uses **8 qubits**: 4 counting qubits to measure the period, and 4 work qubits for a multiply-modulo-15 step. It ends with an inverse quantum Fourier transform (`QFTGate`) and a measurement of the 4 counting qubits.

**An important limitation:** the multiply-modulo-15 step is a known teaching shortcut. It is built from swap and X gates that are hand-chosen for the number 15. That construction uses knowledge of how this special case works. It is not a general factoring program that you could point at an arbitrary integer, let alone a real RSA key. IBM's Shor's algorithm tutorial uses the same kind of compiled N = 15 example.

The next cell only builds and draws the circuit. It does not run it yet.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator
from fractions import Fraction
from math import gcd

N, a = 15, 7      # number to factor, and a helper number we picked
n_count = 4       # qubits used to measure the "period"

def c_amod15(a, power):
    """Builds the step that multiplies by a (mod 15), switched on by one control qubit."""
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        if a in [7, 8]:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        if a in [4, 11]:
            U.swap(1, 3); U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    g = U.to_gate(label=f"{a}^{power} mod 15")
    return g.control()

qc = QuantumCircuit(n_count + 4, n_count)
qc.h(range(n_count))
qc.x(n_count)
for q in range(n_count):
    qc.append(c_amod15(a, 2**q), [q] + [n_count + i for i in range(4)])
qc.append(QFTGate(n_count).inverse(), range(n_count))
qc.measure(range(n_count), range(n_count))

print(qc.draw())

     ┌───┐                                                            »
q_0: ┤ H ├───────■────────────────────────────────────────────────────»
     ├───┤       │                                                    »
q_1: ┤ H ├───────┼──────────────■─────────────────────────────────────»
     ├───┤       │              │                                     »
q_2: ┤ H ├───────┼──────────────┼──────────────■──────────────────────»
     ├───┤       │              │              │                      »
q_3: ┤ H ├───────┼──────────────┼──────────────┼──────────────■───────»
     ├───┤┌──────┴──────┐┌──────┴──────┐┌──────┴──────┐┌──────┴──────┐»
q_4: ┤ X ├┤0            ├┤0            ├┤0            ├┤0            ├»
     └───┘│             ││             ││             ││             │»
q_5: ─────┤1            ├┤1            ├┤1            ├┤1            ├»
          │  7^1 mod 15 ││  7^2 mod 15 ││  7^4 mod 15 ││  7^8 mod 15 │»
q_6: ─────┤2            ├┤2            ├┤2            ├┤2       

## Simulator run (AerSimulator, 1,024 shots)

This cell runs the circuit on a local, noiseless simulator. In a clean run you should see only four bit strings, each about 250 times out of 1,024:

- `0100`, `1000`, and `1100` should produce the factors 3 × 5 after the classical step.
- `0000` is an expected miss. The circuit landed on a valid period-finding outcome, but that particular outcome does not give a factor.

If every shot lands on those four strings, we will later call that **100% on the expected outcomes**. That does *not* mean every shot factored 15, because `0000` is included in the four expected peaks.

In [ ]:
sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print("Measured counts:", counts)

for bits, c in sorted(counts.items(), key=lambda x: -x[1]):
    phase = int(bits, 2) / 2**n_count
    r = Fraction(phase).limit_denominator(N).denominator
    if pow(a, r, N) != 1 and pow(a, 2 * r, N) == 1:
        r *= 2  # the measured fraction can hide a factor of 2 in the period
    if r % 2 == 0 and pow(a, r // 2, N) != N - 1:
        f1 = gcd(a**(r//2) - 1, N)
        if f1 not in (1, N):
            print(f"Result {bits}: phase {phase}, period r = {r}, factors = {f1} x {N // f1}")
            continue
    print(f"Result {bits}: phase {phase}, period guess {r} (no factor from this one)")

Measured counts: {'0100': 259, '0000': 255, '1000': 221, '1100': 289}
Result 1100: phase 0.75, period r = 4, factors = 3 x 5
Result 0100: phase 0.25, period r = 4, factors = 3 x 5
Result 0000: phase 0.0, period guess 1 (no factor from this one)
Result 1000: phase 0.5, period r = 4, factors = 3 x 5


## Hardware run on IBM Quantum (bonus)

This cell sends the same circuit to real IBM hardware.

- Account and **Open Plan** instance on the IBM Quantum Platform (`channel="ibm_quantum_platform"`).
- API key read from a Colab Secret.
- `QiskitRuntimeService` chose a busy-but-available real backend; this job ran on **`ibm_fez`**.
- The circuit was compiled with `generate_preset_pass_manager` at `optimization_level=3`. After compiling, the circuit depth was **4,194** layers.
- `SamplerV2` ran **1,024 shots**.
- Job ID: **`dbpvbbtt7fqc738a2o0g`**.

Expect a messy histogram. Today's hardware adds errors, and a 4,194-layer circuit is already deep. If the four expected answers are still the most common, but many other bit strings appear too, that is a successful honest result, not a failure.

If this cell already has output, **do not run it again** just to polish the notebook. A new run would create a new job ID. If the output disappeared but the job still exists, use the optional "retrieve existing job" cell later in this file instead of submitting a new job.

In [ ]:
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=userdata.get("IBM_API_KEY"))
backend = service.least_busy(operational=True, simulator=False)
print("Running on real hardware:", backend.name)

pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_qc = pm.run(qc)
print("Circuit depth after compiling for this chip:", isa_qc.depth())

job = Sampler(mode=backend).run([isa_qc], shots=1024)
print("Job ID:", job.job_id())
hw_counts = job.result()[0].data.c.get_counts()
print("Hardware counts:", dict(sorted(hw_counts.items(), key=lambda x: -x[1])))

qiskit_runtime_service._discover_account:WARNING:2026-11-11 04:57:12,979: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-11-11 04:57:16,184: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Quant. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-11-11 04:57:16,647: Loading instance: Quant, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-11 04:57:17,892: Using instance: Quant, plan: open


Running on real hardware: ibm_fez


qiskit_runtime_service.backends:WARNING:2026-11-11 04:57:18,591: Using instance: Quant, plan: open


Circuit depth after compiling for this chip: 4194
Job ID: dbpvbbtt7fqc738a2o0g
Hardware counts: {'0100': 100, '1100': 92, '1000': 92, '0000': 87, '1001': 75, '1101': 70, '1011': 60, '0001': 58, '1111': 57, '1110': 55, '0011': 53, '0101': 52, '0111': 46, '0010': 45, '0110': 42, '1010': 40}


In [ ]:
correct = {"0000", "0100", "1000", "1100"}
sim_ok = sum(v for k, v in counts.items() if k in correct) / sum(counts.values())
hw_ok = sum(v for k, v in hw_counts.items() if k in correct) / sum(hw_counts.values())
print(f"Simulator: {sim_ok:.0%} of shots landed on a correct answer")
print(f"Real hardware ({backend.name}): {hw_ok:.0%} of shots landed on a correct answer")
print("A random guess would land on a correct answer 25% of the time (4 of 16 outcomes).")

Simulator: 100% of shots landed on a correct answer
Real hardware (ibm_fez): 36% of shots landed on a correct answer
A random guess would land on a correct answer 25% of the time (4 of 16 outcomes).


## Results

These figures come from the run saved in this notebook on the night of November 10, 2026.

| Run | Where it ran | Shots | Share of shots on the 4 expected outcomes (`0000`, `0100`, `1000`, `1100`) | What else to notice |
| --- | --- | --- | --- | --- |
| Simulator | Qiskit `AerSimulator` | 1,024 | **100%** | Only those four outcomes appeared, each about 250 times. Three of them gave factors 3 × 5. `0000` was the expected miss. |
| Real hardware | IBM **`ibm_fez`**, job `dbpvbbtt7fqc738a2o0g` | 1,024 | **36%** | A uniform random guess among 16 bit strings would hit one of those four about **25%** of the time. After compiling, the circuit was 4,194 layers deep. |

Visible hardware counts from the notebook output (other bit strings were present but not all visible in the screenshot):

`0100`: 100, `1100`: 92, `1000`: 92, `0000`: 87, `1001`: 75, `1101`: 70, `1011`: 60, `0001`: 58, `1111`: 57.

The four expected answers were still the top four, but they only barely stood out from the noise. The hardware did better than chance, yet most shots did not land on an expected outcome.

**What I take from that:** if a tiny, specially compiled circuit for the number 15 already becomes this noisy on current hardware, factoring a real RSA number is still very far away. That is not a reason to ignore the issue. It is a reason to plan a migration while there is still time, because encrypted clinic data copied today may need to stay private for many years.

## Limitations

- The inventory is synthetic. It is not a scan of a real clinic or of MDC systems.
- The priority numbers are a transparent team rule, not an official standard. If we drop a factor, the mid-ranked order can change.
- The years and migration-ease labels are assumptions I chose for the demo.
- The quantum circuit is a compiled N = 15 teaching example. It uses a hand-built multiply step, so it is not evidence that general Shor's algorithm was implemented at full strength.
- A successful factoring of 15 does not mean real RSA or elliptic-curve cryptography can be broken today.
- I am not claiming quantum advantage, and I am not predicting when a cryptographically relevant quantum computer will exist.
- The hardware result is noisy, as expected for a deep circuit on current devices.
- I am not telling anyone to turn off RSA or elliptic-curve cryptography today without a replacement plan. The recommendation is a planned, tested migration, using a hybrid approach where a vendor supports it.
- NIST's "deprecated after 2030" and "disallowed after 2035" statements apply to NIST standards. The earlier federal high-impact dates (key establishment by December 31, 2030, and signatures by December 31, 2031) are background from NIST IR 8547. They are not compliance requirements for this fictional clinic.

## Conclusion

A small organization can start post-quantum planning without waiting for a quantum computer that can break real keys. The practical first step is a clear inventory, a ranking rule other people can check, and a phased roadmap:

- **Now:** internet-facing and partner key exchange, plus encrypted email — look toward FIPS 203 (ML-KEM), and FIPS 204 (ML-DSA) where signatures are also involved.
- **Next:** remaining key-exchange and signature systems, including the RSA wrap around AES-256 backups.
- **Later:** AES-only storage and short-lived internal access. Review key management, but do not treat AES-256 as the same problem as RSA.

The quantum demo supports the "why," honestly: the simulator shows the mechanism, and the `ibm_fez` run shows how noisy even a tiny example is on today's hardware. The work that matters for a clinic is the migration.
